## Section 1: Setup

In [18]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

try:
    import duckdb
except ImportError:
    pip_install("duckdb")
    import duckdb

try:
    import pandas as pd
except ImportError:
    pip_install("pandas")
    import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    pip_install("matplotlib")
    import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    pip_install("seaborn")
    import seaborn as sns

import os
import requests
import zipfile
import io

con = duckdb.connect()
print("DuckDB connected:", duckdb.__version__)

DuckDB connected: 1.5.2


## Section 2: Confirm column names

In [19]:
diag_schema = con.execute("DESCRIBE SELECT * FROM 'data/diagnoses_icd.csv' LIMIT 1").df()
print(diag_schema)

required = {"subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"}
found = set(diag_schema["column_name"].tolist())
missing = required - found
if missing:
    raise RuntimeError(f"Missing required columns in diagnoses_icd.csv: {missing}")
print("\nAll required columns confirmed.")

   column_name column_type null   key default extra
0   subject_id      BIGINT  YES  None    None  None
1      hadm_id      BIGINT  YES  None    None  None
2      seq_num      BIGINT  YES  None    None  None
3     icd_code     VARCHAR  YES  None    None  None
4  icd_version      BIGINT  YES  None    None  None

All required columns confirmed.


## Section 3: Download CCSR ICD-10 Mapping

In [ ]:
import os, requests, zipfile, glob as _glob

CCSR_ZIP_URL = "https://hcup-us.ahrq.gov/toolssoftware/ccsr/DXCCSR_v2025-1.zip"
CCSR_ZIP_PATH = "ccsr/DXCCSR_v2025-1.zip"
CCSR_EXTRACT_DIR = "ccsr/icd10_ccsr"

os.makedirs("ccsr", exist_ok=True)
os.makedirs(CCSR_EXTRACT_DIR, exist_ok=True)

if not os.path.exists(CCSR_ZIP_PATH):
    print("Downloading CCSR ICD-10 mapping from AHRQ...")
    resp = requests.get(CCSR_ZIP_URL, timeout=120)
    resp.raise_for_status()
    with open(CCSR_ZIP_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded: {len(resp.content) / 1024:.1f} KB")
else:
    print(f"Already exists: {CCSR_ZIP_PATH}")

if not _glob.glob(f"{CCSR_EXTRACT_DIR}/*.csv"):
    with zipfile.ZipFile(CCSR_ZIP_PATH, "r") as zf:
        zf.extractall(CCSR_EXTRACT_DIR)
    print("Extracted.")
else:
    print("Already extracted.")

print("\nFiles in ccsr/icd10_ccsr/:")
for f in os.listdir(CCSR_EXTRACT_DIR):
    print(" ", f)

## Section 3b: Download CCS ICD-9 Mapping

In [ ]:
CCS_ZIP_URL = "https://hcup-us.ahrq.gov/toolssoftware/ccs/Single_Level_CCS_2015.zip"
CCS_ZIP_PATH = "ccsr/ccs_icd9.zip"
CCS_EXTRACT_DIR = "ccsr/icd9_ccs"

os.makedirs(CCS_EXTRACT_DIR, exist_ok=True)

if not os.path.exists(CCS_ZIP_PATH):
    print("Downloading CCS ICD-9 mapping from AHRQ...")
    resp = requests.get(CCS_ZIP_URL, timeout=120)
    resp.raise_for_status()
    with open(CCS_ZIP_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded: {len(resp.content) / 1024:.1f} KB")
else:
    print(f"Already exists: {CCS_ZIP_PATH}")

if not _glob.glob(f"{CCS_EXTRACT_DIR}/*.csv"):
    with zipfile.ZipFile(CCS_ZIP_PATH, "r") as zf:
        zf.extractall(CCS_EXTRACT_DIR)
    print("Extracted.")
else:
    print("Already extracted.")

ccs_csvs = _glob.glob(f"{CCS_EXTRACT_DIR}/*.csv")
if not ccs_csvs:
    raise RuntimeError("No CSV found in ccsr/icd9_ccs/ after extraction.")

ccs_csv_path = ccs_csvs[0]
print(f"\nUsing: {ccs_csv_path}")

ccs_raw = pd.read_csv(ccs_csv_path, skiprows=1, nrows=3)
print("\nActual columns after skipping note row:")
print(ccs_raw.columns.tolist())
print("\nFirst 3 rows:")
print(ccs_raw)
con.register("ccs_icd9_map_raw", pd.read_csv(ccs_csv_path, skiprows=1))
con.execute("CREATE OR REPLACE TABLE ccs_icd9_map AS SELECT * FROM ccs_icd9_map_raw")
print(f"\nccs_icd9_map loaded: {con.execute('SELECT COUNT(*) FROM ccs_icd9_map').fetchone()[0]:,} rows")

## Section 4: Load CCSR ICD-10 Mapping into DuckDB

In [ ]:
ccsr_csvs = _glob.glob("ccsr/icd10_ccsr/DXCCSR*.csv")
if not ccsr_csvs:
    raise RuntimeError("No DXCCSR*.csv found in ccsr/icd10_ccsr/")

ccsr_csv_path = ccsr_csvs[0]
print(f"Using: {ccsr_csv_path}")

ccsr_raw = pd.read_csv(ccsr_csv_path, nrows=3)
print("\nActual columns:")
print(ccsr_raw.columns.tolist())
print("\nFirst 3 rows:")
print(ccsr_raw)

# Column 0: ICD-10-CM CODE (join key)
# Column 1: ICD-10-CM CODE DESCRIPTION (free text, skip)
# Column 2: Default CCSR CATEGORY IP (the actual category)
icd10_col = ccsr_raw.columns[0]
ccsr_cat_col = ccsr_raw.columns[2]
print(f"\nJoin key column  : '{icd10_col}'")
print(f"Category column  : '{ccsr_cat_col}'")

ccsr_full = pd.read_csv(ccsr_csv_path)
ccsr_full[icd10_col] = ccsr_full[icd10_col].astype(str).str.strip().str.replace("'", "", regex=False)
con.register("ccsr_map_raw", ccsr_full)
con.execute("CREATE OR REPLACE TABLE ccsr_map AS SELECT * FROM ccsr_map_raw")
print(f"ccsr_map loaded: {con.execute('SELECT COUNT(*) FROM ccsr_map').fetchone()[0]:,} rows")

## Section 5: Filter Diagnoses to Cohort

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE cohort_diagnoses AS
    SELECT d.subject_id, d.hadm_id, d.seq_num, d.icd_code, d.icd_version
    FROM 'data/diagnoses_icd.csv' d
    INNER JOIN 'outputs/cohort_10k.csv' c ON d.subject_id = c.subject_id
""")

total_rows = con.execute("SELECT COUNT(*) FROM cohort_diagnoses").fetchone()[0]
distinct_patients = con.execute("SELECT COUNT(DISTINCT subject_id) FROM cohort_diagnoses").fetchone()[0]
print(f"Total diagnosis rows for cohort : {total_rows:,}")
print(f"Distinct patients with >= 1 diag: {distinct_patients:,}")
print()
print("ICD version breakdown:")
print(con.execute("""
    SELECT icd_version, COUNT(*) AS count
    FROM cohort_diagnoses
    GROUP BY icd_version
    ORDER BY icd_version
""").df().to_string(index=False))

## Section 6: Map ICD Codes to Clinical Categories

In [ ]:
# Step 1: Map ICD-10 via CCSR
con.execute(f"""
    CREATE OR REPLACE TABLE icd10_mapped AS
    SELECT d.subject_id, d.hadm_id, d.icd_code, d.icd_version,
        m."{ccsr_cat_col}" AS ccsr_category
    FROM cohort_diagnoses d
    LEFT JOIN ccsr_map m ON d.icd_code = m."{icd10_col}"
    WHERE d.icd_version = 10
""")
icd10_ok = con.execute("SELECT COUNT(*) FROM icd10_mapped WHERE ccsr_category IS NOT NULL").fetchone()[0]
icd10_no = con.execute("SELECT COUNT(*) FROM icd10_mapped WHERE ccsr_category IS NULL").fetchone()[0]
print(f"ICD-10 mapped  : {icd10_ok:,}")
print(f"ICD-10 unmapped: {icd10_no:,}")
print()

# Step 2: Map ICD-9 via CCS
ccs_cols = con.execute("DESCRIBE ccs_icd9_map").df()["column_name"].tolist()
print(f"CCS ICD-9 columns: {ccs_cols}")
ccs_icd9_code_col = ccs_cols[0]
ccs_cat_candidates = [c for c in ccs_cols if "DESCRIPTION" in c.upper() or "CATEGORY" in c.upper()]
if not ccs_cat_candidates:
    raise RuntimeError(f"Cannot find category column. Columns: {ccs_cols}")
ccs_cat_col_icd9 = ccs_cat_candidates[0]
print(f"ICD-9 code column   : '{ccs_icd9_code_col}'")
print(f"ICD-9 category column: '{ccs_cat_col_icd9}'")
print()

con.execute(f"""
    CREATE OR REPLACE TABLE icd9_mapped AS
    SELECT d.subject_id, d.hadm_id, d.icd_code, d.icd_version,
        m."{ccs_cat_col_icd9}" AS ccsr_category
    FROM cohort_diagnoses d
    LEFT JOIN (
        SELECT
            TRIM(REPLACE(REPLACE("{ccs_icd9_code_col}", chr(39), ''), '"', '')) AS icd9_clean,
            "{ccs_cat_col_icd9}"
        FROM ccs_icd9_map
    ) m ON d.icd_code = m.icd9_clean
    WHERE d.icd_version = 9
""")
icd9_ok = con.execute("SELECT COUNT(*) FROM icd9_mapped WHERE ccsr_category IS NOT NULL").fetchone()[0]
icd9_no = con.execute("SELECT COUNT(*) FROM icd9_mapped WHERE ccsr_category IS NULL").fetchone()[0]
print(f"ICD-9 mapped  : {icd9_ok:,}")
print(f"ICD-9 unmapped: {icd9_no:,}")
print()

# Step 3: Combine into all_mapped
con.execute("""
    CREATE OR REPLACE TABLE all_mapped AS
    SELECT * FROM icd10_mapped
    UNION ALL
    SELECT * FROM icd9_mapped
""")
total_ok = con.execute("SELECT COUNT(*) FROM all_mapped WHERE ccsr_category IS NOT NULL").fetchone()[0]
total_no = con.execute("SELECT COUNT(*) FROM all_mapped WHERE ccsr_category IS NULL").fetchone()[0]
total = total_ok + total_no
rate = total_ok / total * 100 if total > 0 else 0
print(f"Total mapped rows  : {total_ok:,}")
print(f"Total unmapped rows: {total_no:,}")
print(f"Overall mapping rate: {rate:.1f}%")

## Section 7: Build Patient-Level Feature Table

In [ ]:
categories = con.execute("""
    SELECT DISTINCT ccsr_category
    FROM all_mapped
    WHERE ccsr_category IS NOT NULL
    ORDER BY ccsr_category
""").df()["ccsr_category"].tolist()
print(f"Distinct categories found: {len(categories)}")

mapped_df = con.execute("""
    SELECT subject_id, ccsr_category
    FROM all_mapped
    WHERE ccsr_category IS NOT NULL
""").df()

pivot = (
    mapped_df
    .assign(val=1)
    .drop_duplicates(subset=["subject_id", "ccsr_category"])
    .pivot(index="subject_id", columns="ccsr_category", values="val")
    .fillna(0)
    .astype(int)
    .reset_index()
)
pivot.columns.name = None

cohort_ids = con.execute("SELECT DISTINCT subject_id FROM 'outputs/cohort_10k.csv'").df()
pivot = cohort_ids.merge(pivot, on="subject_id", how="left").fillna(0)
pivot["subject_id"] = pivot["subject_id"].astype(int)
feature_cols = [c for c in pivot.columns if c != "subject_id"]
pivot[feature_cols] = pivot[feature_cols].astype(int)

print(f"Feature table shape: {pivot.shape}")
print(f"Expected: (9996, 530 to 600)")

## Section 8: Save as Parquet

In [ ]:
os.makedirs("outputs", exist_ok=True)

parquet_path = "outputs/ccsr_features.parquet"
preview_path = "outputs/ccsr_features_preview.csv"

pivot.to_parquet(parquet_path, index=False)
pivot.head(20).to_csv(preview_path, index=False)

print(f"Saved: {parquet_path} ({os.path.getsize(parquet_path) / 1024:.1f} KB)")
print(f"Saved: {preview_path} ({os.path.getsize(preview_path) / 1024:.1f} KB)")
assert os.path.exists(parquet_path), "Parquet missing"
assert os.path.exists(preview_path), "Preview CSV missing"
print("Both files confirmed on disk.")